# L2b: Errors, Tests, and Debugging a Numerical Program

A program can run to completion, raise nothing, and still be wrong. This lab uses a freight-loading calculation with a unit defect to tell the difference between code that fails loudly and code that fails quietly.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Tell three failure modes apart:__ Distinguish code that will not run, code that raises on bad input, and code that runs successfully and returns a physically wrong answer. Only the third kind needs a reference case to detect.
> * __Localize a defect with a known case:__ Use a case whose answer you already know, together with the ratio between expected and observed results, to find where a calculation went wrong before changing any code.
> * __Implement a defensive interface:__ Write a function that converts units correctly and rejects input the model cannot support, and confirm the repair with a test set that would have caught the original defect.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [4]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

LoadError: LoadError: failed to find source of parent package: "LoopVectorization"
in expression starting at /Users/williammanno/CHEME-5800-CourseRepository-Fall-2026/Include.jl:22
in expression starting at /Users/williammanno/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/Include.jl:36

Besides Julia's `Base` library, this lab uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/). `Include.jl` also pulls in [`src/Compute.jl`](src/Compute.jl), which is where the function you are about to write lives.

___

## A result that looks plausible

A dock supervisor needs to know how long a trailer will occupy a loading bay. The calculation is the obvious one: time equals cargo divided by loading rate, $t = m/r$. Our interface accepts cargo mass in tonnes and the loading rate in kilograms per minute, and returns minutes.

> __The reference case:__
>
> A $2\;\mathrm{t}$ load is $2000\;\mathrm{kg}$. At a sustained $250\;\mathrm{kg/min}$ it takes $2000/250 = 8$ minutes to load. We know the answer before we write any code, which is exactly what makes this case useful.

Here is a first attempt. It compiles, it runs, and it raises nothing:

In [ ]:
function buggy_loading_time_minutes(cargo_tonnes, loading_rate_kg_min)
    return cargo_tonnes / loading_rate_kg_min
end

buggy_result = buggy_loading_time_minutes(2.0, 250.0)

The cell ran and produced a number. That is the trap: nothing announced a problem. Compare what came back against what we know the answer to be, and look at the ratio rather than the difference:

In [ ]:
expected_minutes = 8.0
diagnostic = (
    result = buggy_result,
    expected = expected_minutes,
    passes = isapprox(buggy_result, expected_minutes),
    missing_scale_factor = expected_minutes / buggy_result,
)

The ratio is exactly $1000$, and $1000$ is not a number that appears anywhere in the formula. It is the conversion between tonnes and kilograms.

> __Why the ratio and not the difference?__
>
> A difference tells you the answer is wrong. A ratio often tells you _why_. Clean powers of ten point at unit conversions; a factor of $2$ or $\tfrac{1}{2}$ points at a boundary or an off-by-one; a factor near $\pi$ points at degrees versus radians. Reading the ratio first is faster than reading the code first.

___

## Implement the repair

Now write the real function. It lives in [`src/Compute.jl`](src/Compute.jl) rather than in a cell here, so that it can be imported, tested by the validation suite, and edited with real tooling. Right now that file holds a signature, a docstring, and two `TODO` comments.

> __What to write:__
>
> * __Validate both arguments.__ Throw an [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) naming the argument when `cargo_tonnes` or `loading_rate_kg_min` is not finite, or is not strictly positive. [The `isfinite(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.isfinite) does the first check.
> * __Convert, then divide.__ The defect above was dividing tonnes by kilograms per minute. Convert the cargo mass to kilograms first, then return the quotient as a `Float64`.

Write the validation out in full rather than factoring it into a helper; the whole point is that this one function body is the interface's defence.

Open the file, complete both `TODO`s, then restart the kernel and run this notebook from the top. Until you do, the next cell stops with a "not implemented yet" error, which is the expected starting state.

In [ ]:
correct_result = cargo_loading_time_minutes(2.0, 250.0)
(correct_result = correct_result, agrees = correct_result == expected_minutes)

___

## Exceptions are part of the interface

Invalid input should fail at the interface, with a message that names what was wrong. A caller who passes a zero loading rate deserves to be told which argument was rejected, not handed an `Inf`.

Catch an error only where the program can make a meaningful decision about it. Here we catch it just to display the message:

In [ ]:
caught_message = try
    cargo_loading_time_minutes(2.0, 0.0)
    "no error"
catch error
    sprint(showerror, error)
end

___

## Optional: the same contract in Python

This section is not required to finish the lab. It is here for students who want to see that a contract is a design decision rather than a Julia feature.

The companion Python implementation raises [a `TypeError`](https://docs.python.org/3/library/exceptions.html#TypeError) for the wrong _kind_ of input and [a `ValueError`](https://docs.python.org/3/library/exceptions.html#ValueError) for an invalid numerical _value_. Julia splits those two cases differently: the `::Real` annotation means a `String` argument never reaches the body at all, so Julia raises [a `MethodError`](https://docs.julialang.org/en/v1/base/base/#Core.MethodError) before any of our validation runs, while a bad numerical value does reach our checks and raises [an `ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError). The type system does the first job; the function body does the second.

> __A gap worth noticing:__ [The `Bool` type](https://docs.julialang.org/en/v1/base/numbers/#Core.Bool) is a subtype of `Real` in Julia, so `cargo_loading_time_minutes(true, 100)` satisfies the annotation, passes both checks, and returns `10.0` as though one tonne had been ordered. A contract is only as tight as the types and the explicit checks make it together, which is the same trap `L2d` guards against for `Bool` indices.

The syntax and the exception taxonomy differ; the unit contract and the reference case do not.

In [ ]:
python_source = joinpath(CHEME5800_L2B_ROOT, "src", "cargo_loading.py")
python_preview = join(first(split(read(python_source, String), '\n'), 18), "\n")
python_preview

Run the Python comparison from the bundle root:

```bash
python -m unittest discover -s weeks/week-02/L2b/src -p 'test_*.py'
```

___

## Turn the diagnosis into regression tests

A repair that is not tested is a repair that comes back. These tests pin the reference case, confirm the original defect is gone, and check that invalid input still raises.

Do they all pass?

In [ ]:
@testset "errors, tests, and debugging" begin
    @test buggy_result != expected_minutes
    @test diagnostic.missing_scale_factor == 1000.0
    @test correct_result == expected_minutes
    @test cargo_loading_time_minutes(0.5, 100) == 5.0
    @test_throws ArgumentError cargo_loading_time_minutes(0, 100)
    @test_throws ArgumentError cargo_loading_time_minutes(1, Inf)
    @test occursin("loading_rate_kg_min", caught_message)
end

___

## Summary
A calculation that runs without complaint has proved only that it runs; correctness is a separate claim that needs separate evidence.

> __Key Takeaways:__
>
> * **Silence is not correctness:** A program can execute cleanly and return a physically wrong answer, so the absence of an exception is not evidence that a result is right.
> * **A known case localizes the defect:** Comparing against a result you already know, and reading the ratio rather than the difference, points at the cause before you start reading code.
> * **Validation belongs at the interface:** Rejecting non-finite and non-positive input where the caller crosses into your function is what stops a bad value from propagating into results that look plausible.

Every calculation you write this semester will have a unit contract, whether or not you write it down. Writing it down is what lets a test catch the day you break it.
___